# RAG Vector Search Setup

Set up Vector Search infrastructure for the RAG (Retrieval-Augmented Generation) agent used in the workshop.

## What This Notebook Does
1. Creates a Unity Catalog schema and Delta table for document chunks
2. Creates a Vector Search endpoint (may take 5-15 minutes if new)
3. Reads and chunks markdown policy documents
4. Loads document chunks into the Delta table
5. Creates a Delta Sync index with managed embeddings
6. Verifies the setup with a test similarity query

## Prerequisites
- Run `00b_setup_data` first (loads data into Unity Catalog)
- Databricks workspace with Unity Catalog enabled
- Permission to create Vector Search endpoints

## 1. Install Dependencies

In [ ]:
%pip install databricks-vectorsearch langchain-text-splitters --quiet
dbutils.library.restartPython()

## 2. Setup and Configuration

In [ ]:
# Setup and Environment Check
import os
import sys

IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IN_DATABRICKS:
    # Check for Unity Catalog support
    try:
        uc_enabled = spark.conf.get("spark.databricks.unityCatalog.enabled", "false")
        if uc_enabled.lower() != "true":
            print("WARNING: Unity Catalog may not be enabled.")
            print("This notebook requires Unity Catalog (not available in Community Edition).")
            print("Please use Databricks Free Trial or a workspace with Unity Catalog.")
    except Exception:
        pass

    # Path setup for imports
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = "/Workspace" + "/".join(notebook_path.split("/")[:-2])
    if workspace_path not in sys.path:
        sys.path.insert(0, workspace_path)
else:
    project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

print("Environment ready!")
print(f"Running in Databricks: {IN_DATABRICKS}")

# Configuration Widgets
if IN_DATABRICKS:
    try:
        dbutils.widgets.removeAll()
    except Exception:
        pass

    dbutils.widgets.text("1_catalog", "workshop", "1. Catalog")
    dbutils.widgets.text("2_schema", "rag", "2. Schema")
    dbutils.widgets.text("3_endpoint_name", "rag-workshop-endpoint", "3. VS Endpoint")
    dbutils.widgets.text("4_embedding_model", "databricks-bge-large-en", "4. Embedding Model")

    print("\nConfigure using the widgets above, then run the next cells.")
    print("")
    print("Widget options:")
    print("  1. Catalog: Unity Catalog name (must already exist from 00b_setup_data)")
    print("  2. Schema: Schema for RAG tables (will be created)")
    print("  3. VS Endpoint: Vector Search endpoint name")
    print("  4. Embedding Model: Databricks embedding model endpoint")
else:
    print("Running locally - using default values.")
    print("Modify the execution cell to change configuration.")

## 3. Core Functions

In [ ]:
# Core Functions for RAG Vector Search Setup
import glob as glob_module
import hashlib
import json
import time

TABLE_NAME = "document_chunks"
INDEX_NAME = "document_index"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200


def create_schema_and_table(catalog, schema):
    """Create Unity Catalog schema and Delta table with Change Data Feed enabled."""
    full_table = f"{catalog}.{schema}.{TABLE_NAME}"

    # Create schema
    print(f"  Creating schema: {catalog}.{schema}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

    # Create table with CDF enabled (required for Delta Sync indexes)
    print(f"  Creating table: {full_table}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {full_table} (
            id STRING NOT NULL,
            content STRING NOT NULL,
            source STRING NOT NULL,
            metadata STRING
        )
        USING DELTA
        TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)

    # Ensure CDF is enabled even if table already existed
    spark.sql(f"""
        ALTER TABLE {full_table}
        SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)


def get_or_create_endpoint(endpoint_name):
    """Create Vector Search endpoint if it does not exist, then wait for ONLINE status."""
    from databricks.vector_search.client import VectorSearchClient

    vsc = VectorSearchClient()

    # Check if endpoint already exists
    try:
        endpoints = vsc.list_endpoints()
        existing = [e for e in endpoints.get("endpoints", []) if e.get("name") == endpoint_name]

        if existing:
            status = existing[0].get("endpoint_status", {}).get("state", "UNKNOWN")
            if status == "ONLINE":
                print(f"  Endpoint '{endpoint_name}' already exists and is ONLINE. Skipping creation.")
                return
            else:
                print(f"  Endpoint '{endpoint_name}' exists (status: {status}). Waiting for ONLINE...")
    except Exception:
        pass  # Assume endpoint doesn't exist
    else:
        if not existing:
            # Create new endpoint
            print(f"  Creating endpoint '{endpoint_name}' (this may take 5-15 minutes)...")
            vsc.create_endpoint(
                name=endpoint_name,
                endpoint_type="STANDARD",
            )

    # Poll for ONLINE status
    max_wait = 900  # 15 minutes
    wait_interval = 30
    elapsed = 0

    while elapsed < max_wait:
        try:
            endpoint = vsc.get_endpoint(endpoint_name)
            status = endpoint.get("endpoint_status", {}).get("state", "UNKNOWN")

            if status == "ONLINE":
                return
            elif status in ("FAILED", "DELETED"):
                raise RuntimeError(f"Endpoint creation failed with status: {status}")

            print(f"  ... Endpoint status: {status} (waited {elapsed}s)")
        except RuntimeError:
            raise
        except Exception as e:
            print(f"  ... Checking status: {e}")

        time.sleep(wait_interval)
        elapsed += wait_interval

    raise TimeoutError(
        f"Endpoint '{endpoint_name}' did not reach ONLINE status within {max_wait}s. "
        "Check the Databricks UI for status."
    )


def chunk_documents(docs_path):
    """Read markdown files and split them into chunks using RecursiveCharacterTextSplitter."""
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
    )

    # Find all markdown files
    md_files = sorted(glob_module.glob(os.path.join(docs_path, "*.md")))
    if not md_files:
        raise FileNotFoundError(f"No markdown files found in {docs_path}")

    all_chunks = []
    for file_path in md_files:
        filename = os.path.basename(file_path)
        with open(file_path, encoding="utf-8") as f:
            content = f.read()

        texts = splitter.split_text(content)
        print(f"  {filename}: {len(content):,} chars -> {len(texts)} chunks")

        for i, text in enumerate(texts):
            hash_input = f"{filename}:{i}:{text[:100]}"
            chunk_id = hashlib.sha256(hash_input.encode()).hexdigest()[:16]

            all_chunks.append(
                {
                    "id": chunk_id,
                    "content": text.strip(),
                    "source": filename,
                    "metadata": json.dumps(
                        {
                            "position": i,
                            "total_chunks": len(texts),
                            "char_count": len(text),
                        }
                    ),
                }
            )

    return all_chunks


def load_chunks_to_table(catalog, schema, chunks):
    """Load document chunks into the Delta table using Spark DataFrame."""
    full_table = f"{catalog}.{schema}.{TABLE_NAME}"

    # Truncate existing data for idempotent re-runs
    print(f"  Clearing existing data in {full_table}...")
    spark.sql(f"TRUNCATE TABLE {full_table}")

    # Create DataFrame from chunks and write to table
    from pyspark.sql import Row

    rows = [Row(**chunk) for chunk in chunks]
    df = spark.createDataFrame(rows)
    df.write.mode("append").saveAsTable(full_table)

    # Return row count for verification
    count = spark.sql(f"SELECT COUNT(*) FROM {full_table}").first()[0]
    return count


def create_or_get_index(endpoint_name, catalog, schema, embedding_model):
    """Create Delta Sync index with managed embeddings if it does not already exist."""
    from databricks.vector_search.client import VectorSearchClient

    vsc = VectorSearchClient()

    source_table = f"{catalog}.{schema}.{TABLE_NAME}"
    index_name = f"{catalog}.{schema}.{INDEX_NAME}"

    # Check if index already exists
    try:
        index = vsc.get_index(endpoint_name=endpoint_name, index_name=index_name)
        print(f"  Index '{index_name}' already exists. Triggering sync...")
        index.sync()
        print("  Sync triggered. Embeddings will update in the background.")
        return
    except Exception:
        pass  # Index doesn't exist, create it

    # Create Delta Sync index with managed embeddings
    print(f"  Creating index '{index_name}'...")
    print(f"  Source table: {source_table}")
    print(f"  Embedding model: {embedding_model}")

    vsc.create_delta_sync_index(
        endpoint_name=endpoint_name,
        source_table_name=source_table,
        index_name=index_name,
        pipeline_type="TRIGGERED",
        primary_key="id",
        embedding_source_column="content",
        embedding_model_endpoint_name=embedding_model,
    )

    # Wait for index to be ready
    print("  Waiting for index to be ready...")
    max_wait = 600  # 10 minutes
    wait_interval = 30
    elapsed = 0

    while elapsed < max_wait:
        try:
            index = vsc.get_index(endpoint_name=endpoint_name, index_name=index_name)
            status = index.describe().get("status", {}).get("ready", False)
            if status:
                print("  Index is ready. Triggering initial sync...")
                index.sync()
                return
            print(f"  ... Index not ready yet (waited {elapsed}s)")
        except Exception as e:
            print(f"  ... Checking index: {e}")

        time.sleep(wait_interval)
        elapsed += wait_interval

    print("  Index creation may still be in progress. Check the Databricks UI.")
    print("  You can also trigger a sync manually from the Vector Search UI.")


print("Core functions loaded!")

## 4. Execute Setup

This cell will:
1. Create the Unity Catalog schema and Delta table
2. Create or connect to the Vector Search endpoint
3. Read, chunk, and load markdown documents
4. Create the Delta Sync index with managed embeddings

In [ ]:
# Execute RAG Vector Search Setup

if IN_DATABRICKS:
    catalog = dbutils.widgets.get("1_catalog")
    schema = dbutils.widgets.get("2_schema")
    endpoint_name = dbutils.widgets.get("3_endpoint_name")
    embedding_model = dbutils.widgets.get("4_embedding_model")
else:
    catalog = "workshop"
    schema = "rag"
    endpoint_name = "rag-workshop-endpoint"
    embedding_model = "databricks-bge-large-en"

print("=" * 60)
print("RAG Vector Search Setup")
print("=" * 60)
print(f"  Catalog:         {catalog}")
print(f"  Schema:          {schema}")
print(f"  Endpoint:        {endpoint_name}")
print(f"  Embedding Model: {embedding_model}")
print()

if not IN_DATABRICKS:
    print("NOTICE: Not running in Databricks. Skipping actual setup.")
    print("This notebook is designed to run in Databricks with Unity Catalog.")
else:
    # Step 1: Schema and Table
    print("--- Step 1: Creating schema and table ---")
    create_schema_and_table(catalog, schema)
    print(f"  Table {catalog}.{schema}.{TABLE_NAME} ready")

    # Step 2: VS Endpoint
    print("\n--- Step 2: Setting up Vector Search endpoint ---")
    get_or_create_endpoint(endpoint_name)
    print(f"  Endpoint '{endpoint_name}' is ONLINE")

    # Step 3: Chunk and Load Documents
    print("\n--- Step 3: Loading documents ---")
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = "/Workspace" + "/".join(notebook_path.split("/")[:-2])
    docs_path = os.path.join(workspace_path, "data", "documents")

    chunks = chunk_documents(docs_path)
    print(f"  Chunked {len(set(c['source'] for c in chunks))} documents into {len(chunks)} chunks")

    count = load_chunks_to_table(catalog, schema, chunks)
    print(f"  Loaded {count} chunks into {catalog}.{schema}.{TABLE_NAME}")

    # Step 4: VS Index
    print("\n--- Step 4: Creating Vector Search index ---")
    create_or_get_index(endpoint_name, catalog, schema, embedding_model)
    print(f"  Index {catalog}.{schema}.{INDEX_NAME} ready")

    print("\n" + "=" * 60)
    print("Setup complete!")
    print("=" * 60)

## 5. Verify Setup

Check that documents were loaded and the index is responding to queries.

In [ ]:
# Verify RAG Setup

if IN_DATABRICKS:
    from databricks.vector_search.client import VectorSearchClient

    vsc = VectorSearchClient()

    # Check 1: Table row count
    full_table = f"{catalog}.{schema}.{TABLE_NAME}"
    count = spark.sql(f"SELECT COUNT(*) FROM {full_table}").first()[0]
    print(f"Document chunks in table: {count}")

    # Check 2: Documents loaded
    sources = spark.sql(f"SELECT DISTINCT source FROM {full_table} ORDER BY source").collect()
    print(f"\nDocuments loaded ({len(sources)}):")
    for row in sources:
        doc_count = spark.sql(f"SELECT COUNT(*) FROM {full_table} WHERE source = '{row.source}'").first()[0]
        print(f"  - {row.source} ({doc_count} chunks)")

    # Check 3: Max chunk length (should be close to CHUNK_SIZE)
    max_len = spark.sql(f"SELECT MAX(LENGTH(content)) FROM {full_table}").first()[0]
    print(f"\nMax chunk length: {max_len} characters")

    # Check 4: Test similarity query
    index_name = f"{catalog}.{schema}.{INDEX_NAME}"
    try:
        index = vsc.get_index(endpoint_name=endpoint_name, index_name=index_name)
        results = index.similarity_search(
            query_text="What is the vehicle warranty policy?",
            columns=["content", "source"],
            num_results=3,
        )
        print(f"\nTest query results ({results.get('result', {}).get('row_count', 0)} matches):")
        if results.get("result", {}).get("data_array"):
            for row in results["result"]["data_array"]:
                print(f"  Source: {row[1]}")
                print(f"  Preview: {row[0][:100]}...")
                print()
    except Exception as e:
        print(f"\nIndex may still be syncing. Try again in a few minutes: {e}")
else:
    print("Verification requires Databricks environment. Skipping.")

## 6. Next Steps

Use these values in `01_agent_basics` notebook to enable RAG:

| Widget | Value |
|--------|-------|
| **vector_search_endpoint** | _(your endpoint name)_ |
| **vector_search_index** | _(catalog.schema.document_index)_ |

Set `mock_mode` to `false` to use real Vector Search instead of mock responses.

In [ ]:
# Print configuration values for use in other notebooks

if IN_DATABRICKS:
    print("Configure 01_agent_basics with these values:")
    print(f"  vector_search_endpoint = {endpoint_name}")
    print(f"  vector_search_index    = {catalog}.{schema}.{INDEX_NAME}")
    print("\nOr set environment variables:")
    print(f"  VECTOR_SEARCH_ENDPOINT={endpoint_name}")
    print(f"  VECTOR_SEARCH_INDEX={catalog}.{schema}.{INDEX_NAME}")
else:
    print("Run this notebook in Databricks to see configuration values.")

## 7. Cleanup (Optional)

Uncomment and run the cells below to remove all RAG infrastructure created by this notebook.

**Warning:** This will permanently delete the Vector Search index, endpoint, and document data.

In [ ]:
# CLEANUP - Uncomment to remove RAG infrastructure
# WARNING: This will delete the Vector Search index, endpoint, and data!

# if IN_DATABRICKS:
#     from databricks.vector_search.client import VectorSearchClient
#     vsc = VectorSearchClient()
#
#     catalog = dbutils.widgets.get("1_catalog")
#     schema = dbutils.widgets.get("2_schema")
#     endpoint_name = dbutils.widgets.get("3_endpoint_name")
#
#     # Delete index first
#     try:
#         vsc.delete_index(endpoint_name=endpoint_name, index_name=f"{catalog}.{schema}.{INDEX_NAME}")
#         print(f"Deleted index: {catalog}.{schema}.{INDEX_NAME}")
#     except Exception as e:
#         print(f"Index deletion: {e}")
#
#     # Delete endpoint
#     try:
#         vsc.delete_endpoint(endpoint_name)
#         print(f"Deleted endpoint: {endpoint_name}")
#     except Exception as e:
#         print(f"Endpoint deletion: {e}")
#
#     # Drop table and schema
#     spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.{TABLE_NAME}")
#     print(f"Dropped table: {catalog}.{schema}.{TABLE_NAME}")
#     spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")
#     print(f"Dropped schema: {catalog}.{schema}")